# Quant Finance – Learning Notebook
# Module 3: Strategies, Backtesting & Instruments

**Author:** Florian Ebner

## Source References
- de Prado, M.L. (2018). *Advances in Financial Machine Learning*. Wiley.
- Chan, E. (2013). *Algorithmic Trading*. Wiley.
- Engle, R. & Granger, C. (1987). *Co-Integration and Error Correction*. Econometrica.
- Jegadeesh, N. & Titman, S. (1993). *Returns to Buying Winners and Selling Losers*. Journal of Finance.
- Hull, J.C. (2018). *Options, Futures, and Other Derivatives* (10th ed.). Pearson.
- Fabozzi, F.J. (2007). *Fixed Income Analysis* (2nd ed.). CFA Institute.
---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import minimize

np.random.seed(42)
plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': '#f8f9fa',
    'axes.grid': True, 'grid.color': '#e0e0e0',
    'font.family': 'sans-serif', 'axes.spines.top': False, 'axes.spines.right': False,
})
TRADING_DAYS = 252
RISK_FREE    = 0.035
print('Module 3 – Strategies, Backtesting & Instruments')

---
## 15. Mean Reversion Strategies

### Theory

**Mean reversion** describes the tendency of prices or spreads to return to their historical mean after deviating from it.

**Mathematical foundation – Ornstein-Uhlenbeck Process:**
$$dX_t = \kappa (\theta - X_t) dt + \sigma dW_t$$

- $\kappa$: **Mean-reversion speed** (how quickly does X return to θ?). Half-life: $t_{1/2} = \ln(2)/\kappa$
- $\theta$: Long-run mean
- $\sigma$: Volatility of shocks

### When Does Mean Reversion Apply?

- **Spreads** between related assets (pairs trading)
- **Currency pairs** (purchasing power parity as long-run anchor)
- **Interest rates** (short rates revert to long-run equilibrium)
- **Volatility** (VIX reverts to historical mean ~20)
- **Equities** on short timeframes (1–5 days) are often mean-reverting

### Warning

**Do not confuse with random walk:** Most individual price series are **not** mean-reverting processes on their own. Mean reversion typically appears in **price differentials** or **relative valuations** (e.g. P/E ratios), not in absolute prices.

**Source:** Chan (2013), Ch. 2; Avellaneda, M. & Lee, J. (2010). *Statistical Arbitrage in the US Equities Market*. Quantitative Finance

In [ ]:
def simulate_ou(kappa, theta, sigma, x0, T_years, dt=1/252):
    n    = int(T_years / dt)
    x    = np.zeros(n)
    x[0] = x0
    for t in range(1, n):
        dW   = np.random.normal(0, np.sqrt(dt))
        x[t] = x[t-1] + kappa * (theta - x[t-1]) * dt + sigma * dW
    return x

kappa_fast = 50.0
kappa_slow = 5.0
T_ou       = 2.0

ou_fast = simulate_ou(kappa_fast, 0, 0.05, 0.1, T_ou)
ou_slow = simulate_ou(kappa_slow, 0, 0.05, 0.1, T_ou)

spread    = pd.Series(ou_fast)
roll_mean = spread.rolling(20).mean()
roll_std  = spread.rolling(20).std()
upper     = roll_mean + 1.5 * roll_std
lower     = roll_mean - 1.5 * roll_std
signal    = np.where(spread > upper, -1, np.where(spread < lower, 1, 0))
signal    = pd.Series(signal).shift(1).fillna(0)

print(f'Half-life (Fast κ={kappa_fast}): {np.log(2)/kappa_fast*TRADING_DAYS:.1f} trading days')
print(f'Half-life (Slow κ={kappa_slow}):  {np.log(2)/kappa_slow*TRADING_DAYS:.1f} trading days')

fig, axes = plt.subplots(2, 1, figsize=(13, 8))

axes[0].plot(ou_fast, color='#e74c3c', lw=1,
             label=f'Fast κ={kappa_fast} (HL={np.log(2)/kappa_fast*TRADING_DAYS:.0f}d)')
axes[0].plot(ou_slow, color='#3498db', lw=1.5,
             label=f'Slow κ={kappa_slow} (HL={np.log(2)/kappa_slow*TRADING_DAYS:.0f}d)')
axes[0].axhline(0, color='black', lw=1, ls='--', label='Long-run mean θ=0')
axes[0].set_title('Ornstein-Uhlenbeck Process: Reversion Speed Comparison', fontweight='bold')
axes[0].legend()

axes[1].plot(spread.values, color='#4a90d9', lw=1, label='Spread')
axes[1].plot(roll_mean.values, color='black', lw=1.5, ls='--', label='Rolling Mean')
axes[1].plot(upper.values, color='#e74c3c', lw=1, ls=':', label='+1.5σ (Short Signal)')
axes[1].plot(lower.values, color='#2ecc71', lw=1, ls=':', label='-1.5σ (Long Signal)')
buy_idx  = np.where(signal == 1)[0]
sell_idx = np.where(signal == -1)[0]
axes[1].scatter(buy_idx,  spread.iloc[buy_idx],  color='#2ecc71', s=30, zorder=5, label='Long')
axes[1].scatter(sell_idx, spread.iloc[sell_idx], color='#e74c3c', s=30, zorder=5, label='Short')
axes[1].set_title('Bollinger-Band Mean-Reversion Strategy', fontweight='bold')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('mean_reversion.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 16. Momentum and Trend Following

### Theory

**Momentum** describes the empirically observed tendency: **assets that have performed well in the past tend to continue performing well short-term.** This is the opposite of mean reversion.

**Jegadeesh & Titman (1993):** The most well-known evidence: equity winners of the past 3–12 months outperform over the next 3–12 months. Momentum premium ~7–10% p.a. historically, but highly crisis-prone ("momentum crashes").

### Forms of Momentum

- **Cross-Sectional Momentum (CS-Mom):** Buy the best 20% of stocks, sell the worst 20%. Relative between assets.
- **Time-Series Momentum (TS-Mom):** Buy when an asset is above its historical mean, regardless of its relative performance. Also known as: **trend following**.

### Technical Indicators for Trend Following

- **Moving Average Crossover:** When short-term MA (e.g. 50 days) crosses above long-term MA (200 days) → buy signal ("Golden Cross")
- **MACD (Moving Average Convergence Divergence)**
- **Breakout systems:** Buy when price makes a new 52-week high

### Why Does Momentum Work?

Behavioural finance: **underreaction** to corporate news (investors adjust expectations too slowly) and **herding**. Not a free lunch — momentum strategies suffer periodic extreme losses ("momentum crashes") when trends reverse abruptly.

**Source:** Jegadeesh & Titman (1993); Moskowitz, T. et al. (2012). *Time Series Momentum*. Journal of Financial Economics

In [ ]:
T_mom    = 1260
drift    = 0.0003
returns_m = np.zeros(T_mom)
vol_now   = 0.012
for t in range(T_mom):
    vol_now      = np.sqrt(0.00003 + 0.1*(returns_m[t-1]**2 if t>0 else 0.0001) + 0.85*vol_now**2)
    returns_m[t] = drift + vol_now * np.random.randn()

prices_m = pd.Series(100 * np.exp(np.cumsum(returns_m)))
ma_50    = prices_m.rolling(50).mean()
ma_200   = prices_m.rolling(200).mean()
signal_ma = (ma_50 > ma_200).astype(int).shift(1).fillna(0)
ret_m     = prices_m.pct_change().fillna(0)
strat_r   = signal_ma * ret_m
cum_bh    = (1 + ret_m).cumprod()
cum_strat = (1 + strat_r).cumprod()

def perf(r, label):
    cum  = (1+r).cumprod()
    cagr = cum.iloc[-1]**(TRADING_DAYS/T_mom) - 1
    vol  = r.std() * np.sqrt(TRADING_DAYS)
    sr   = (cagr - RISK_FREE) / vol
    mdd  = ((cum - cum.cummax())/cum.cummax()).min()
    print(f'  {label}: CAGR={cagr*100:.2f}%, Vol={vol*100:.2f}%, SR={sr:.3f}, MDD={mdd*100:.2f}%')

print('MA Crossover vs. Buy & Hold:')
perf(ret_m,   'Buy & Hold')
perf(strat_r, 'MA(50/200)')

fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

axes[0].plot(prices_m.values, color='#4a90d9', lw=1, alpha=0.8, label='Price')
axes[0].plot(ma_50.values,    color='#e67e22', lw=1.5, label='MA 50')
axes[0].plot(ma_200.values,   color='#c0392b', lw=2,   label='MA 200')
axes[0].fill_between(range(T_mom), prices_m.min(), prices_m.max(),
                     where=signal_ma==1, alpha=0.1, color='green')
axes[0].set_title('MA(50/200) Crossover Strategy', fontweight='bold')
axes[0].set_ylabel('Price')
axes[0].legend()

axes[1].plot(cum_bh.values,    color='#95a5a6', lw=2, label='Buy & Hold')
axes[1].plot(cum_strat.values, color='#2ecc71', lw=2, label='MA Strategy')
axes[1].set_title('Cumulative Value: Strategy vs. Buy & Hold', fontweight='bold')
axes[1].set_ylabel('Cumulative Value')
axes[1].legend()

plt.tight_layout()
plt.savefig('momentum.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 17. Statistical Arbitrage & Pairs Trading with Cointegration

### Theory

**Statistical Arbitrage (StatArb)** exploits **statistically predictable price relationships** between assets without relying on fundamental analysis.

**Pairs Trading:** The simplest form of StatArb:
1. Find two assets that historically move together (e.g. Volkswagen and BMW)
2. When the spread between them becomes too large: buy the undervalued, sell the overvalued
3. When the spread normalises: close both positions

### Cointegration vs. Correlation

**Important distinction:** Two stocks can be highly correlated (both rising) but not cointegrated. Cointegration means: there exists a **linear combination** of both series that is **stationary** (mean-reverting):

$$\text{Spread}_t = P_{A,t} - \beta \cdot P_{B,t} \sim \text{stationary}$$

**Engle-Granger Test (1987):**
1. Estimate $\beta$ via OLS: $P_A = \alpha + \beta P_B + \epsilon$
2. Test whether residuals $\epsilon$ are stationary (ADF test)
3. If stationary → cointegration → spread is mean-reverting → pairs trade viable

**Source:** Engle & Granger (1987); Gatev, E. et al. (2006). *Pairs Trading: Performance of a Relative-Value Arbitrage Rule*. Review of Financial Studies

In [ ]:
T_pair = 756

common_trend = np.cumsum(np.random.normal(0, 0.005, T_pair))
beta_true    = 1.5
noise_B      = np.random.normal(0, 0.004, T_pair)

price_A = 100 * np.exp(common_trend + 0.3 * simulate_ou(20, 0, 0.03, 0, T_pair/TRADING_DAYS))
price_B = 100 * np.exp(common_trend / beta_true + np.cumsum(noise_B))
prices_A = pd.Series(price_A)
prices_B = pd.Series(price_B)

beta_hat_p, alpha_hat_p = np.polyfit(prices_B.values, prices_A.values, 1)
spread_p = prices_A - beta_hat_p * prices_B - alpha_hat_p

delta_spread = spread_p.diff().dropna().values
spread_lag   = spread_p.shift(1).dropna().values
adf_beta, _, _, adf_p, _ = stats.linregress(spread_lag, delta_spread)

print(f'Estimated Hedging Ratio β : {beta_hat_p:.4f}  (true: {beta_true:.1f})')
print(f'ADF coefficient (γ)       : {adf_beta:.6f}  (negative = mean-reversion)')
print(f'ADF p-value               : {adf_p:.6f}')
print(f'Cointegration             : {"YES ✓" if adf_p < 0.05 else "NO ✗"}  (p < 0.05)')

spread_mean = spread_p.rolling(60).mean()
spread_std  = spread_p.rolling(60).std()
z_score     = (spread_p - spread_mean) / spread_std
signal_pairs = pd.Series(np.where(z_score < -1, 1, np.where(z_score > 1, -1, 0)))
signal_pairs = signal_pairs.shift(1).fillna(0)
ret_A_p      = prices_A.pct_change().fillna(0)
ret_B_p      = prices_B.pct_change().fillna(0)
strat_ret_p  = signal_pairs * (ret_A_p - beta_hat_p * ret_B_p)
cum_pairs    = (1 + strat_ret_p).cumprod()

fig, axes = plt.subplots(3, 1, figsize=(13, 11))

ax2 = axes[0].twinx()
axes[0].plot(prices_A.values, color='#4a90d9', lw=1.5, label='Asset A')
ax2.plot(prices_B.values,     color='#e67e22', lw=1.5, label='Asset B')
axes[0].set_title('Cointegrated Assets: Shared Trend, Mean-Reverting Spread', fontweight='bold')
axes[0].set_ylabel('Asset A Price', color='#4a90d9')
ax2.set_ylabel('Asset B Price', color='#e67e22')

axes[1].plot(spread_p.values,    color='#2c3e50', lw=1.5, label='Spread')
axes[1].plot(spread_mean.values, color='grey',    lw=1, ls='--', label='Rolling Mean')
axes[1].fill_between(range(len(spread_p)), spread_mean - spread_std,
                     spread_mean + spread_std, alpha=0.15, color='blue', label='±1σ')
long_p  = signal_pairs == 1
short_p = signal_pairs == -1
axes[1].scatter(np.where(long_p)[0],  spread_p.iloc[long_p].values,  color='green', s=15, zorder=5)
axes[1].scatter(np.where(short_p)[0], spread_p.iloc[short_p].values, color='red',   s=15, zorder=5)
axes[1].set_title('Spread (= P_A - β·P_B): Mean-Reverting', fontweight='bold')
axes[1].legend()

axes[2].plot(cum_pairs.values, color='#2ecc71', lw=2)
axes[2].axhline(1, color='grey', ls='--', lw=1)
axes[2].set_title('Pairs Trade Cumulative P&L (Market Neutral)', fontweight='bold')
axes[2].set_ylabel('Cumulative Value')

plt.tight_layout()
plt.savefig('pairs_trading.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 18. Backtesting Biases: Survivorship Bias & Look-Ahead Bias

### Theory

The **most serious mistakes** in quantitative finance research arise from flawed backtests. Seemingly profitable strategies are often just the result of methodological errors.

### Survivorship Bias

**Definition:** When a backtest includes only companies that *still exist* at the time of analysis, firms that went bankrupt, merged, or were delisted are excluded. This biases performance statistics upward.

**Example:** A backtest of DAX stocks from 2024 going back to 2000 includes only companies still in the DAX in 2024. Companies that went insolvent during that period (e.g. Wirecard) are missing. The simulated portfolio therefore implicitly contains perfect foresight — which stocks will survive the next 24 years.

**Quantification:** Studies estimate survivorship bias in equity returns at **1–3% p.a.**

### Look-Ahead Bias

**Definition:** The strategy uses information at a historical point t that was only available after t.

**Common forms:**
1. **Data point leak:** Using the closing price to signal a trade on the same day
2. **Parameter leak:** Normalising using mean/std of the entire dataset instead of only historical data
3. **Index composition:** Using today's index composition for historical backtest periods
4. **Reporting delays:** Financial data is often only available weeks after quarter end

**Prevention:** **Point-in-time databases** (Compustat, Bloomberg) store data as it was available at each historical point.

**Source:** de Prado (2018), Ch. 11; Harvey, C. et al. (2016). *...and the Cross-Section of Expected Returns*. Review of Financial Studies

In [ ]:
n_stocks  = 100
T_surv    = 252 * 5
mu_cross  = 0.06 / TRADING_DAYS
sig_cross = 0.20 / np.sqrt(TRADING_DAYS)

all_returns = np.random.normal(mu_cross, sig_cross, (T_surv, n_stocks))
cum_all     = np.exp(np.cumsum(all_returns, axis=0))

bankrupt_mask = (cum_all < 0.20)
survived      = ~bankrupt_mask.any(axis=0)
n_survived    = survived.sum()

all_final_ret     = (cum_all[-1] - 1)
mean_ret_all      = all_final_ret.mean()
mean_ret_survived = all_final_ret[survived].mean()

print('Survivorship Bias Demonstration:')
print(f'  Total stocks                  : {n_stocks}')
print(f'  Survived (5 years)            : {n_survived} ({n_survived/n_stocks*100:.1f}%)')
print(f'  Bankrupt                      : {n_stocks-n_survived}')
print(f'  Avg 5Y Return (ALL)           : {mean_ret_all*100:.2f}%')
print(f'  Avg 5Y Return (SURVIVORS ONLY): {mean_ret_survived*100:.2f}%')
print(f'  Survivorship Bias             : {(mean_ret_survived-mean_ret_all)*100:.2f}%')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(all_final_ret[~survived]*100, bins=20, color='#e74c3c', alpha=0.7,
             label=f'Bankrupt ({(~survived).sum()} stocks)', edgecolor='white')
axes[0].hist(all_final_ret[survived]*100,  bins=20, color='#2ecc71', alpha=0.7,
             label=f'Survived ({n_survived} stocks)', edgecolor='white')
axes[0].axvline(mean_ret_all*100,      color='#c0392b', lw=2, ls='--',
                label=f'All Avg={mean_ret_all*100:.1f}%')
axes[0].axvline(mean_ret_survived*100, color='#27ae60', lw=2, ls='--',
                label=f'Survived Avg={mean_ret_survived*100:.1f}%')
axes[0].set_xlabel('5-Year Total Return (%)')
axes[0].set_title('Survivorship Bias:\nSurvivors Show Upward-Biased Returns', fontweight='bold')
axes[0].legend(fontsize=9)

cum_survived_mean = cum_all[:, survived].mean(axis=1)
cum_all_mean      = cum_all.mean(axis=1)
axes[1].plot(cum_survived_mean, color='#2ecc71', lw=2, label='Avg (survivors only)')
axes[1].plot(cum_all_mean,      color='#e74c3c', lw=2, label='Avg (all incl. bankrupt)')
axes[1].axhline(1, color='grey', ls='--', lw=1)
axes[1].set_title('Survivorship Bias: Divergence Over Time', fontweight='bold')
axes[1].set_ylabel('Cumulative Value')
axes[1].legend()

plt.tight_layout()
plt.savefig('survivorship_bias.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 19. Walk-Forward Analysis & Out-of-Sample Testing

### Theory

**In-Sample (IS) vs. Out-of-Sample (OOS):**

- **In-Sample:** Data used for parameter estimation/optimisation. Any strategy optimised on IS data adapts to noise (**overfitting**).
- **Out-of-Sample:** Data *never* used during strategy development. OOS performance is the only valid estimate of true strategy performance.

**Walk-Forward Analysis** is the most robust backtesting methodology:
1. Split data into rolling windows
2. **Training Window:** Optimise parameters on this window
3. **Test Window:** Apply optimised parameters to the *next* window (OOS)
4. Slide the window forward and repeat

The result is a complete OOS performance over the entire dataset — without ever looking into the future.

**OOS/IS Performance Ratio:** If a strategy has an IS Sharpe of 2.0 but OOS Sharpe of only 0.5, that is a strong overfitting signal. Rule of thumb: OOS performance should be at least 50% of IS performance.

**Source:** de Prado (2018), Ch. 11 & 12; Chan (2013), Ch. 3

In [ ]:
T_wf         = 252 * 7
train_window = 252
test_window  = 63

r_wf     = pd.Series(np.random.normal(0.0003, 0.011, T_wf))
price_wf = (1 + r_wf).cumprod()

wf_returns_is  = []
wf_returns_oos = []

t = train_window
while t + test_window <= T_wf:
    train_r = r_wf.iloc[t-train_window:t]
    test_r  = r_wf.iloc[t:t+test_window]
    train_p = price_wf.iloc[t-train_window:t]
    test_p  = price_wf.iloc[t:t+test_window]

    best_sr, best_ma = -np.inf, 20
    for ma_len in [10, 20, 30, 50, 100]:
        if ma_len >= len(train_p): continue
        ma      = train_p.rolling(ma_len).mean()
        sig     = (train_p > ma).astype(int).shift(1).fillna(0)
        strat_r = sig * train_r
        if strat_r.std() > 0:
            sr = strat_r.mean() / strat_r.std() * np.sqrt(TRADING_DAYS)
            if sr > best_sr:
                best_sr, best_ma = sr, ma_len

    ma_oos  = pd.concat([train_p.iloc[-best_ma:], test_p]).rolling(best_ma).mean().iloc[best_ma:]
    sig_oos = (test_p.values > ma_oos.values).astype(int)
    oos_r   = pd.Series(np.roll(sig_oos, 1) * test_r.values)

    wf_returns_is.append(best_sr)
    wf_returns_oos.append(oos_r.mean() / oos_r.std() * np.sqrt(TRADING_DAYS)
                          if oos_r.std() > 0 else 0)
    t += test_window

is_sr  = np.array(wf_returns_is)
oos_sr = np.array(wf_returns_oos)

print(f'Walk-Forward Analysis – {len(is_sr)} windows:')
print(f'  Avg Sharpe IS  (In-Sample)     : {is_sr.mean():.3f}')
print(f'  Avg Sharpe OOS (Out-of-Sample) : {oos_sr.mean():.3f}')
print(f'  OOS/IS Ratio                   : {oos_sr.mean()/is_sr.mean():.2f}')
print(f'  (Rule of thumb: OOS > 50% of IS = no extreme overfitting)')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(is_sr, oos_sr, alpha=0.6, color='#4a90d9', s=60)
axes[0].axline([0,0], [1,1], color='red', ls='--', lw=1, label='IS=OOS (perfect)')
axes[0].axhline(0, color='black', lw=0.7)
axes[0].axvline(0, color='black', lw=0.7)
axes[0].set_xlabel('IS Sharpe Ratio')
axes[0].set_ylabel('OOS Sharpe Ratio')
axes[0].set_title('IS vs. OOS Sharpe: Walk-Forward', fontweight='bold')
axes[0].legend()

x = np.arange(len(is_sr))
axes[1].bar(x - 0.2, is_sr,  0.35, label='IS Sharpe',  color='#4a90d9', alpha=0.8)
axes[1].bar(x + 0.2, oos_sr, 0.35, label='OOS Sharpe', color='#2ecc71', alpha=0.8)
axes[1].axhline(0, color='black', lw=0.7)
axes[1].set_title('IS vs. OOS Sharpe per Window', fontweight='bold')
axes[1].set_xlabel('Window Number')
axes[1].set_ylabel('Sharpe Ratio')
axes[1].legend()

plt.tight_layout()
plt.savefig('walk_forward.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 20. Transaction Costs and Slippage

### Theory

The most common reason why backtested-profitable strategies fail in live trading: **transaction costs were underestimated.**

**Components of Transaction Costs:**

1. **Explicit Costs – Commissions:** Broker fees. For institutional investors: 0.02–0.10% per trade.

2. **Bid-Ask Spread:** The difference between the buy (ask) and sell (bid) price. For large-caps (DAX): typically 0.01–0.05%. For small-caps or illiquid assets: 0.1–1%+. Every trade costs half the spread.

3. **Market Impact / Slippage:** For large orders, your own order moves the market. If you buy 1% of daily traded volume, you push the price up (square-root market impact model: $\Delta P \approx \sigma \sqrt{Q/ADV}$ where ADV = Average Daily Volume).

4. **Opportunity Cost:** Price moves during order execution.

**Break-Even Turnover:**
$$\text{Max Turnover} = \frac{\text{Gross Alpha}}{TC}$$

A strategy with 1% gross alpha and 0.1% TC can turn over its capital at most 10x per year — anything more eats the return.

**Source:** de Prado (2018), Ch. 14; Almgren, R. & Chriss, N. (2001). *Optimal Execution of Portfolio Transactions*. Journal of Risk

In [ ]:
T_tc = 252 * 3
r_tc = pd.Series(np.random.normal(0.0005, 0.012, T_tc))
signal_noisy = pd.Series(np.random.normal(0, 1, T_tc))
signal_tc    = np.sign(signal_noisy).shift(1).fillna(0)
signal_changes   = signal_tc.diff().abs() / 2
annual_turnover  = signal_changes.mean() * TRADING_DAYS
gross_ret        = signal_tc * r_tc

tc_levels = [0.0, 0.0005, 0.001, 0.002, 0.005]
tc_labels = ['0 bp (no TC)', '5 bp', '10 bp', '20 bp', '50 bp']

print(f'Annual Turnover: {annual_turnover:.1f}x')
print(f'{"TC":>12} {"Net SR":>10} {"Annual TC Cost":>18}')
print('-' * 43)

fig, ax = plt.subplots(figsize=(12, 5))
colors_tc = ['#2ecc71', '#3498db', '#f1c40f', '#e67e22', '#e74c3c']

for tc, label, col in zip(tc_levels, tc_labels, colors_tc):
    net_ret  = gross_ret - signal_changes * tc
    cum_net  = (1 + net_ret).cumprod()
    ann_ret  = cum_net.iloc[-1] ** (TRADING_DAYS/T_tc) - 1
    ann_vol  = net_ret.std() * np.sqrt(TRADING_DAYS)
    sr_net   = (ann_ret - RISK_FREE) / ann_vol if ann_vol > 0 else 0
    tc_costs = tc * signal_changes.sum() / T_tc * TRADING_DAYS * 100
    print(f'{label:>12} {sr_net:>10.3f} {tc_costs:>17.3f}%')
    ax.plot(cum_net.values, color=col, lw=1.8, label=f'{label} (SR={sr_net:.2f})')

ax.axhline(1, color='grey', ls='--', lw=0.8)
ax.set_title(f'Impact of Transaction Costs  |  Turnover={annual_turnover:.0f}x p.a.', fontweight='bold')
ax.set_ylabel('Cumulative Value')
ax.set_xlabel('Trading Days')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('transaction_costs.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 21. Bond Pricing, Yields and the Yield Curve

### Theory

A **bond** pays regular **coupons** (interest payments) and at maturity returns the **face value**. The price of a bond is the **present value of all future cash flows**:

$$P = \sum_{t=1}^{T} \frac{C}{(1+y)^t} + \frac{N}{(1+y)^T}$$

- $C$: Coupon payment (= coupon rate × face value)
- $N$: Face value (par value), typically €1,000
- $y$: **Yield to Maturity (YTM)** — the single discount rate that equates all cash flows to the current price
- $T$: Maturity in years

### Inverse Price-Yield Relationship

**When yields rise → bond prices fall.** This is the most fundamental principle of the bond market:
- If an existing bond pays a 3% coupon and new bonds offer 5%, the old bond is less attractive → price falls
- The YTM of a bond moves in step with the market price

### Duration and Modified Duration

**Macaulay Duration:** Weighted average maturity of cash flows (in years)

$$D_{Mac} = \frac{\sum_{t=1}^T t \cdot \frac{CF_t}{(1+y)^t}}{P}$$

**Modified Duration:** Interest rate sensitivity of the bond price:
$$D_{Mod} = \frac{D_{Mac}}{1+y} \approx -\frac{1}{P} \frac{dP}{dy}$$

**Interpretation:** A Modified Duration of 7 means: if the yield rises by 1% (100 bps), the bond price falls by ~7%.

### Yield Curve

The **yield curve** shows YTMs for different maturities (3M, 1Y, 2Y, 5Y, 10Y, 30Y) at the same point in time:
- **Normal (upward sloping):** Long-term rates higher than short-term — normal growth expectation
- **Flat:** Transition phase
- **Inverted:** Short-term rates higher — classic recession signal (predicted every US recession since 1960)

**Source:** Fabozzi (2007); Mishkin, F. (2015). *The Economics of Money, Banking, and Financial Markets*

In [ ]:
def bond_price(face, coupon_rate, ytm, years):
    n      = int(years)
    c      = face * coupon_rate
    ytm_p  = ytm
    pv_coupons = c * (1 - (1 + ytm_p)**(-n)) / ytm_p
    pv_face    = face / (1 + ytm_p)**n
    return pv_coupons + pv_face

def macaulay_duration(face, coupon_rate, ytm, years):
    n     = int(years)
    c     = face * coupon_rate
    t_arr = np.arange(1, n + 1)
    cfs   = np.full(n, c)
    cfs[-1] += face
    pvs   = cfs / (1 + ytm) ** t_arr
    price = pvs.sum()
    return (t_arr * pvs).sum() / price

face, coup, yrs = 1000, 0.04, 10
yields_range = np.linspace(0.01, 0.12, 200)
prices_range = [bond_price(face, coup, y, yrs) for y in yields_range]

price_at_4 = bond_price(face, coup, 0.04, yrs)
dur_mac    = macaulay_duration(face, coup, 0.04, yrs)
dur_mod    = dur_mac / (1 + 0.04)

print(f'Bond: Face={face}€, Coupon={coup*100:.0f}%, Maturity={yrs}Y, YTM=4%')
print(f'  Price             : €{price_at_4:.2f}')
print(f'  Macaulay Duration : {dur_mac:.3f} years')
print(f'  Modified Duration : {dur_mod:.3f}')
print(f'  Price at YTM=5%   : €{bond_price(face, coup, 0.05, yrs):.2f}  '
      f'(Δ ≈ {(bond_price(face,coup,0.05,yrs)/price_at_4-1)*100:.2f}%)')
print(f'  Duration estimate : -{dur_mod*0.01*100:.2f}%  (well approximated!)')

maturities     = [0.25, 0.5, 1, 2, 3, 5, 7, 10, 20, 30]
curve_normal   = [3.5, 3.6, 3.8, 4.0, 4.2, 4.5, 4.7, 4.9, 5.1, 5.2]
curve_flat     = [4.5, 4.5, 4.5, 4.5, 4.5, 4.5, 4.5, 4.5, 4.5, 4.5]
curve_inverted = [5.2, 5.1, 4.9, 4.6, 4.3, 4.0, 3.8, 3.6, 3.5, 3.4]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(yields_range * 100, prices_range, color='#4a90d9', lw=2)
axes[0].scatter([4], [price_at_4], color='red', s=100, zorder=5)
axes[0].annotate(f'YTM=4%, P=€{price_at_4:.0f}', (4, price_at_4),
                 textcoords='offset points', xytext=(10, -15), fontsize=9)
axes[0].set_xlabel('Yield to Maturity (%)')
axes[0].set_ylabel('Bond Price (€)')
axes[0].set_title('Inverse Price-Yield Relationship\n(10Y, 4% Coupon, €1000 Face)', fontweight='bold')

axes[1].plot(maturities, curve_normal,   color='#2ecc71', lw=2, marker='o', ms=5, label='Normal (steep)')
axes[1].plot(maturities, curve_flat,     color='#f1c40f', lw=2, marker='s', ms=5, label='Flat')
axes[1].plot(maturities, curve_inverted, color='#e74c3c', lw=2, marker='^', ms=5,
             label='Inverted (recession signal)')
axes[1].set_xlabel('Maturity (years)')
axes[1].set_ylabel('Yield (%)')
axes[1].set_title('Yield Curve Scenarios', fontweight='bold')
axes[1].legend()
plt.tight_layout()
plt.savefig('bonds_yield_curve.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 22. Corporate Actions: Dividends, Splits, Mergers

### Theory

**Corporate actions** are events that change a company's capital structure. If not handled correctly in a backtest, they create **phantom gains or losses** in historical data.

### Dividends

On the **ex-dividend day** the stock price falls by the dividend amount (exactly in theory, approximately in practice). If price data is not **dividend-adjusted**:
- Prices show a sudden drop on the ex-div day
- This generates a false sell signal in momentum systems
- **Solution:** Use **total return** data (adjusted for dividends) or calculate returns including dividend payments

**Adjustment Factor:** All historical prices *before* the ex-div date are multiplied by:
$$\text{Adjustment Factor} = \frac{P_{\text{close, t-1}} - D}{P_{\text{close, t-1}}}$$

### Stock Splits

In a **2:1 split** the number of shares doubles and the price halves. Raw data shows a 50% price crash on the split day — catastrophic for any strategy that looks at absolute prices.

**Split adjustment:** All historical prices *before* the split date are divided by the split factor.

### Rights Issues

When companies raise capital, existing shareholders receive the right to buy new shares at a preferential price. The **Theoretical Ex-Rights Price (TERP)** adjusts the price:
$$TERP = \frac{N \cdot P_{\text{market}} + M \cdot P_{\text{subscription}}}{N + M}$$

**Source:** de Prado (2018), Ch. 2; Hull (2018), Ch. 17

In [ ]:
T_corp   = 252 * 2
base_ret = np.random.normal(0.0003, 0.011, T_corp)
price_raw = pd.Series(100 * np.exp(np.cumsum(base_ret)))

div_day   = 200
div_amt   = 2.0
split_day = 350
split_fac = 2.0

price_unadj = price_raw.copy()
price_unadj.iloc[div_day:]   -= div_amt
price_unadj.iloc[split_day:] /= split_fac

price_adj = price_raw.copy()
adj_factor_div = (price_unadj.iloc[div_day-1] - div_amt) / price_unadj.iloc[div_day-1]
price_adj.iloc[:div_day] *= adj_factor_div
price_adj.iloc[:split_day] /= split_fac

ret_unadj = price_unadj.pct_change().fillna(0)
ret_adj   = price_adj.pct_change().fillna(0)

print('Corporate Actions – Impact on Backtests:')
print(f'  Price before dividend  : €{price_unadj.iloc[div_day-1]:.2f}')
print(f'  Price after  dividend  : €{price_unadj.iloc[div_day]:.2f}  '
      f'(Δ={price_unadj.iloc[div_day]-price_unadj.iloc[div_day-1]:+.2f}€)')
print(f'  Return unadjusted      : {ret_unadj.iloc[div_day]*100:+.2f}%  ← false signal!')
print(f'  Return adjusted        : {ret_adj.iloc[div_day]*100:+.4f}%   ← correct')

fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

axes[0].plot(price_unadj.values, color='#e74c3c', lw=1.5, label='Unadjusted (raw data)')
axes[0].plot(price_adj.values,   color='#2ecc71', lw=1.5, ls='--', label='Adjusted (total return)')
axes[0].axvline(div_day,   color='blue',   lw=1.5, ls=':', label=f'Ex-Dividend Day (Δ=-€{div_amt})')
axes[0].axvline(split_day, color='orange', lw=1.5, ls=':', label='2:1 Split Day')
axes[0].set_title('Corporate Actions: Raw Data vs. Adjusted Data', fontweight='bold')
axes[0].set_ylabel('Price (€)')
axes[0].legend(fontsize=9)

axes[1].plot(ret_unadj.values * 100, color='#e74c3c', lw=0.8, alpha=0.8, label='Unadjusted returns')
axes[1].plot(ret_adj.values   * 100, color='#2ecc71', lw=0.8, alpha=0.8, label='Adjusted returns')
axes[1].axvline(div_day,   color='blue',   lw=1.5, ls=':', label='Ex-Div')
axes[1].axvline(split_day, color='orange', lw=1.5, ls=':', label='Split')
axes[1].set_ylabel('Daily Return (%)')
axes[1].set_xlabel('Trading Days')
axes[1].set_title('False Returns from Unadjusted Corporate Actions', fontweight='bold')
axes[1].legend(fontsize=9)
plt.tight_layout()
plt.savefig('corporate_actions.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Summary – Module 3

| Concept | Key Insight | Practical Relevance |
|---------|------------|---------------------|
| **Mean Reversion** | OU process; half-life = ln(2)/κ | Spread strategies, fixed income |
| **Momentum** | Winners tend to keep winning; Golden Cross | Trend following funds, CTA |
| **Pairs Trading** | Cointegration ≠ correlation; ADF test | StatArb, market-neutral hedge funds |
| **Survivorship Bias** | Only survivors in backtest → +1–3% p.a. phantom alpha | Use point-in-time data |
| **Look-Ahead Bias** | Future data in signal → false alpha | Rolling parameters; point-in-time |
| **Walk-Forward** | OOS performance = only valid estimate | OOS/IS > 50% as rule of thumb |
| **Transaction Costs** | Break-even turnover = Alpha/TC | High frequency costs more; sizing matters |
| **Bond Pricing** | $P = \Sigma CF/(1+y)^t$; Modified Duration | Interest rate sensitivity; DV01 |
| **Yield Curve** | Normal, flat, inverted | Inverted curve = recession signal |
| **Corporate Actions** | Splits/dividends → false returns | Always use total return data |

---
**→ Continue with Module 4: Derivatives, Greeks, Volatility Trading & Performance Attribution**